# 🎓 **Workshop: Fine-Tuning Gemma for KMU Datensouveränität**

**Mittelstand-Digital Zentrum Spreeland**  
**Dozent:** KI-Trainer-Team  
**Ziel:** Ein vortrainiertes Open-Weights-Modell (Gemma / Gemma 4 Series) auf hochspezifische, interne KMU-Maschinendaten und Wartungsprotokolle anpassen, ohne sensible Daten mit externen Cloud-APIs zu teilen.

⚠️ **WICHTIGER SETUP-SCHRITT: T4 GPU LAUFZEIT AKTIVIEREN**  
Da Google Colab standardmäßig oft eine reine CPU-Laufzeit startet, müssen Sie vor dem Ausführen der ersten Code-Zelle manuell eine **T4 GPU** aktivieren:  
1. Klicken Sie im Menü oben auf **Laufzeit (Runtime)** -> **Laufzeittyp ändern (Change runtime type)**.  
2. Wählen Sie unter *Hardwarebeschleuniger (Hardware accelerator)* den Eintrag **T4 GPU** aus.  
3. Klicken Sie auf **Speichern (Save)**.  
4. Verifizieren Sie rechts oben in der Verbindungsanzeige, dass dort `T4` aktiv ist.  

---

## 1. Installation & Umgebung einrichten

Wir nutzen die offiziellen, stabilen Bibliotheken von **Hugging Face** (`transformers`, `peft`, `trl`) und **bitsandbytes** für das 4-Bit QLoRA Fine-Tuning. Diese Methode benötigt keine C++ Compilation, baut keine Wheels aus dem Quellcode und ist daher **zu 100% kompatibel und stabil** in allen Google Colab und Kaggle Umgebungen.

In [ ]:
# Installation der stabilen Standard-HuggingFace-Bibliotheken
!pip install -q -U bitsandbytes datasets accelerate peft trl transformers

## 2. Hugging Face Authentifizierung (Gemma ist ein lizenziertes Modell)

Gemma-Modelle sind **gated Models** von Google. Um darauf zuzugreifen, müssen Sie den Nutzungsbedingungen einmalig zustimmen:
1. Besuchen Sie **[Hugging Face - Google Gemma 2B IT](https://huggingface.co/google/gemma-2b-it)** und akzeptieren Sie die Nutzungsbedingungen.
2. Erstellen Sie einen kostenfreien Lese-Token (Read Token) unter **[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)**.
3. Fügen Sie diesen Token in Google Colab links in den **Geheimnissen (Secrets / Schlüssel-Symbol 🔑)** unter dem Namen `HF_TOKEN` ein und aktivieren Sie den Schalter für den Notebook-Zugriff.

## 3. Base Modell in 4-Bit QLoRA laden

Wir konfigurieren eine hochpräzise 4-Bit-Quantisierung via `BitsAndBytesConfig` (NF4), um den Videospeicher (VRAM) der T4 GPU zu schonen. Als Modell können Sie das standardmäßige `google/gemma-2-2b-it` oder ein anderes Modell aus der Gemma-Familie wählen.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata

# Hugging Face Token aus den Colab-Secrets auslesen
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

# Alternativ hier direkt manuell einsetzen falls keine Secrets genutzt werden:
# hf_token = "hf_YOUR_TOKEN_HERE"

# Sie können hier google/gemma-2-2b-it oder jedes andere Gemma-Modell wählen
model_id = "google/gemma-2-2b-it"

# 4-Bit Quantisierung für die T4 GPU konfigurieren
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # T4 GPU bevorzugt float16
    bnb_4bit_use_double_quant=True
)

print(f"Lade Modell {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},  # Automatisch auf die T4 GPU laden
    token=hf_token
)
print("Modell erfolgreich geladen!")

## 4. LoRA (Low-Rank Adaptation) konfigurieren

Wir bereiten das Modell für das Parameter-Effiziente Fine-Tuning (PEFT) vor und definieren eine LoRA-Konfiguration, die nur bestimmte Gewichtsschichten anpasst.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Modell auf 4-Bit Training vorbereiten
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,                       # Rang der Anpassung
    lora_alpha=16,             # Skalierungsfaktor
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. KMU-Spezifischen Datensatz vorbereiten

Wir simulieren den Ernstfall: Interne Wartungsanweisungen und Fehlermeldungen von Werkstatt-Maschinen der fiktiven *Spreeland-Maschinenbau GmbH* werden als Trainingsdatensatz eingelesen und in das offizielle Gemma-Prompt-Format gebracht.

In [ ]:
import pandas as pd
from datasets import Dataset

# Interne Q&A-Datenbank der Spreeland-Maschinenbau GmbH
kmu_data = [
    {
        "instruction": "Fehlermeldung E104 an der Spreeland Press V4 beheben.",
        "response": "Der Fehler E104 an der Press V4 deutet auf einen unzureichenden Öldruck im Hydraulikkreislauf hin. Gehen Sie wie folgt vor:\n1. Schalten Sie die Maschine ab und sichern Sie sie gegen Wiedereinschalten.\n2. Überprüfen Sie den Füllstand des Hydrauliköls am Schauglas (Soll: min. 60%).\n3. Prüfen Sie den Druckminderer Ventil Y3 auf Verstopfung.\n4. Reinigen Sie den Ölfilter F12. Falls der Fehler bestehen bleibt, kontaktieren Sie den Werkskundendienst."
    },
    {
        "instruction": "Wartungsintervall für den Turbomix S300 Mischkopf.",
        "response": "Der Mischkopf des Turbomix S300 erfordert folgende Wartungsintervalle:\n- Täglich: Sichtprüfung auf Dichtheit der Flansche und Reinigung der Mischkammer mit Spülmittellösung Typ M.\n- Alle 500 Betriebsstunden: Nachschmierung der Hauptlagerwelle mit Hochtemperaturfett Spreeland-HT-2.\n- Alle 2.000 Betriebsstunden oder jährlich: Vollständiger Austausch der PTFE-Dichtungen am Mischwellenauslass."
    },
    {
        "instruction": "Wie starte ich den Kalibrierungsmodus der Spreeland LaserCut 9000?",
        "response": "Um den Kalibrierungsmodus an der LaserCut 9000 zu starten:\n1. Navigieren Sie im Hauptmenü des Bedienpanels auf 'Service' -> 'System-Diagnose'.\n2. Geben Sie die KMU-Service-PIN '7739' ein.\n3. Wählen Sie 'Optische Justierung' und drücken Sie den grünen Taster 'Start/OK'.\n4. Die Maschine fährt nun automatisch die 5 Referenzpunkte ab. Platzieren Sie hierzu die Justierplatte auf dem Schneidtisch."
    }
]

df = pd.DataFrame(kmu_data)
dataset = Dataset.from_pandas(df)

# Gemma Chat template definieren
gemma_prompt = """<|turn>user
{}<turn|>
<|turn>model
{}<turn|>"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["response"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = gemma_prompt.format(instruction, output)
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

## 6. Training konfigurieren & starten

Wir füttern die Daten mithilfe des `SFTTrainer` in das Modell. Da unser Datensatz extrem klein ist, genügen 40 Trainingsschritte (Steps), um dem Modell dieses Spezialwissen anzulernen.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Tokenizer padding konfigurieren
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 512,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 2,
        max_steps = 20,
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 1,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

## 7. Verifizierung & Offline Inferenz

Testen wir nun das Modell! Es antwortet nun haargenau mit den geheimen KMU-Handbuch-Spezifikationen, ohne dass diese jemals ins Internet gesendet wurden.

In [ ]:
model.config.use_cache = True

prompt = gemma_prompt.format(
    "Fehlermeldung E104 an der Spreeland Press V4 beheben.",
    "" # Leer lassen zur Generierung
)

inputs = tokenizer(prompt, return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 8. Speichern des trainierten Adapters

Sie können das trainierte Modell direkt als LoRA-Adapter speichern und im KMU-Betrieb offline nutzen.

In [ ]:
# Speichern der reinen Adapter-Gewichte (nur wenige Megabytes)
model.save_pretrained("spreeland_gemma_adapters")
tokenizer.save_pretrained("spreeland_gemma_adapters")